In [2]:
import pandas as pd
import numpy as np
import yfinance as yf


In [157]:
tickers = [
    "^NDX",
    "^GSPC",
    "^DJI",
    "^RUT",
    "AAPL",
    "MSFT",
    "NVDA",
    "AMZN",
    "GOOGL",
    "META",
    "TSLA",
]

prices = yf.download(tickers=tickers, start ="2010-01-01",multi_level_index=False, auto_adjust=True)
prices = prices["Close"]

prices.isnull().sum()
prices.info()


[*********************100%***********************]  11 of 11 completed

<class 'pandas.DataFrame'>
DatetimeIndex: 4226 entries, 2010-01-04 to 2026-10-21
Data columns (total 11 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   AAPL    4226 non-null   float64
 1   AMZN    4226 non-null   float64
 2   GOOGL   4226 non-null   float64
 3   META    3627 non-null   float64
 4   MSFT    4226 non-null   float64
 5   NVDA    4226 non-null   float64
 6   TSLA    4104 non-null   float64
 7   ^DJI    4226 non-null   float64
 8   ^GSPC   4226 non-null   float64
 9   ^NDX    4226 non-null   float64
 10  ^RUT    4226 non-null   float64
dtypes: float64(11)
memory usage: 396.2 KB


In [ ]:
#R dziennie stope zwrotu
daily_return = prices.pct_change(1).dropna()
daily_return

#ogolny zwrot
total_return = (1+daily_return).prod() - 1


#Zannualizowana zmiennosc / wahania
annualized_volatility= daily_return.std() * (252**0.5)
annualized_volatility


#zannualizowany zwrot
years = len(daily_return) /252
annualized_return = (1 + total_return) ** (1/years) -1
annualized_return

#Wskaznik sharpa
risk_free_rate = 0
sharpe_ratio = ((annualized_return/years) - risk_free_rate)/annualized_volatility
sharpe_ratio

#najwiekszy dołek
rolling_max = prices.cummax()
drawdown = (prices - rolling_max) / rolling_max
drawdown = drawdown.min()

r2 = daily_return.corr()**2

market = daily_return["^NDX"]

def SCL(table):
    company = table
    b , a = np.polyfit(market,company,1)
    tabelka = pd.Series({"alfa": a,"beta" : b})
    return tabelka

scl = daily_return.apply(SCL).T


Ticker
AAPL     0.058725
AMZN     0.053519
GOOGL    0.060338
META     0.039053
MSFT     0.063722
NVDA     0.091728
TSLA     0.054964
^DJI     0.045434
^GSPC    0.055110
^NDX     0.062951
^RUT     0.030827
dtype: float64

In [155]:
summary_table = pd.DataFrame({
    "total_return" : total_return,
    "annualized_volatility" : annualized_volatility,
    "sharpe_ratio" : sharpe_ratio,
    "max_drawdown" : drawdown,
    "scl_alfa": scl["alfa"],
    "scl_beta": scl["beta"]
    })

summary_table

,total_return,annualized_volatility,sharpe_ratio,max_drawdown,scl_alfa,scl_beta
Ticker,,,,,,
AAPL,20.684409,0.282129,0.058725,-0.437972,2.333428e-04,1.001188
AMZN,23.537759,0.323449,0.053519,-0.561453,2.350786e-04,1.106106
GOOGL,21.492175,0.278220,0.060338,-0.443200,2.453339e-04,0.992238
META,17.420103,0.399394,0.039053,-0.767361,2.215788e-04,1.162632
MSFT,22.092164,0.265925,0.063722,-0.371485,2.363337e-04,0.996537
NVDA,802.547452,0.448389,0.091728,-0.663351,1.039780e-03,1.555755
TSLA,209.342931,0.569296,0.054964,-0.736322,9.832291e-04,1.464761
^NDX,11.032341,0.208355,0.062951,-0.355631,-1.221643e-18,1.000000


Ticker
AAPL     0.313221
AMZN     0.343702
GOOGL    0.420283
META     0.588305
MSFT     0.244446
NVDA     1.062513
TSLA     0.400897
^NDX     0.306314
dtype: float64